# Exploración y limpieza de datos

En este notebook analizamos y preparamos el dataset de Spotify para alinearlo con el objetivo del proyecto.

**Dataset original:** [Spotify Tracks Dataset (Kaggle)](https://www.kaggle.com/datasets/yashdev01/spotify-tracks-dataset)

In [1]:
#manipulate the data with a dataframe
import pandas as pd

#math for cleaning and analyzing the data
import numpy as np

#statistics
from scipy import stats 

#graphing and visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

In [2]:
df=pd.read_csv('../data/spotify_tracks.csv')

#Rows and columns
print(df.shape[0])

#First 5 rows of the dataframe
df.head()

114000


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
#Get a concise summary of the dataframe, including the number of non-null entries and data types for each column
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          114000 non-nu

In [4]:
#Delete the first column
#we could have used the index_col parameter in the read_csv function to avoid this step, but we will do it here for demonstration purposes
df = df.drop(columns=['Unnamed: 0'])

In [5]:
#print the row that have null values
df[df['artists'].isnull()]

#Eliminate the rows with null values
#We can use it this time becuase we only have one row with null values, so we will not lose much data by dropping it
df=df.dropna()
print(df.shape[0])

113999


### ¿Qué hace `duplicated`?

`duplicated` marca con `True` las filas que ya aparecieron antes. En este paso comparamos solo `track_id` mediante el parámetro `subset`, y luego usamos `sum()` para contar cuántas filas duplicadas hay.


In [6]:
print(f"filas duplicadas {df.duplicated(subset=['track_id']).sum()}")
print(f"filas totales {len(df)}")

filas duplicadas 24259
filas totales 113999


### Revisión de filas duplicadas

En este paso queremos observar si existe alguna diferencia entre la información de los registros duplicados.

**Hallazgo:** la única diferencia entre duplicados es el género. Por eso no aportan información adicional y pueden introducir sesgo de duplicación.


### Notas sobre el código

- `keep=False` marca **todos** los repetidos como `True`, incluido el primero.
- `drop_duplicates` elimina los repetidos manteniendo el orden de aparición.


In [7]:
duplicados_ids = df[df.duplicated(subset='track_id', keep=False)]['track_id'].unique()
ejemplo_id=duplicados_ids[0]
df[df['track_id']==ejemplo_id]

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
62102,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.0322,0.000001,0.358,0.715,87.917,4,j-pop
99152,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.0322,0.000001,0.358,0.715,87.917,4,singer-songwriter
102151,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.0322,0.000001,0.358,0.715,87.917,4,songwriter


### Eliminación de duplicados

Teniendo en cuenta el posible sesgo detectado, eliminamos las filas duplicadas y conservamos una sola copia por `track_id`.


In [8]:
df_clean = df.drop_duplicates(subset='track_id', keep='first')
print(f"Antes: {len(df)} filas -> Después: {len(df_clean)} filas")

Antes: 113999 filas -> Después: 89740 filas


### Valores corruptos o placeholders

El siguiente paso es buscar valores corruptos o placeholders en el dataset.

Según la documentación:

| Columna | Valor placeholder | Significado |
|---------|-------------------|-------------|
| `key` | `-1` | Tonalidad desconocida o no detectada |
| `tempo` | `0` | Valor imposible para una canción real |


In [9]:
print(f"Canciones con key = -1: {(df_clean['key'] == -1).sum()}")
print(f"Canciones con tempo = 0: {(df_clean['tempo'] == 0).sum()}")

Canciones con key = -1: 0
Canciones con tempo = 0: 157


### Eliminación de `tempo = 0`

Hay **157** canciones con `tempo = 0`. En un `df_clean` de **89.740** filas, representan apenas el **0,17%** del total, por lo que eliminarlas es la mejor decisión.


In [10]:
df_clean = df_clean[df_clean['tempo'] != 0]
print(f"Filas después de eliminar tempo=0: {len(df_clean)}")

Filas después de eliminar tempo=0: 89583


### Revisión de otras columnas

Ahora revisamos el resto de columnas numéricas en busca de valores atípicos o inconsistentes.


In [11]:
cols_numericas = ['popularity', 'duration_ms', 'danceability', 'energy', 
                    'key', 'loudness', 'mode', 'speechiness', 'acousticness', 
                    'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']

# Revisar min y max de todas las columnas numéricas relevantes


for col in cols_numericas:
    print(f"{col}: min={df_clean[col].min()}, max={df_clean[col].max()}")

popularity: min=0, max=100
duration_ms: min=15800, max=5237295
danceability: min=0.0513, max=0.985
energy: min=2.02e-05, max=1.0
key: min=0, max=11
loudness: min=-46.591, max=4.532
mode: min=0, max=1
speechiness: min=0.0221, max=0.965
acousticness: min=0.0, max=0.996
instrumentalness: min=0.0, max=1.0
liveness: min=0.00925, max=1.0
valence: min=0.0, max=0.995
tempo: min=30.2, max=243.372
time_signature: min=0, max=5


### Valor atípico en `duration_ms`

El valor máximo de `duration_ms` es **5.237.295**, lo que equivale a una duración de ~**87 minutos** — inusual para una pista individual.

Investigaremos ese registro antes de decidir si eliminarlo.


In [12]:
df_clean[df_clean['duration_ms']==5237295]

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
73617,3Cnz3Bu9Wcw8p3kiBTXTxp,Tale Of Us,Unity (Voyage Mix),Unity (Voyage Mix) Pt. 1,35,5237295,False,0.695,0.736,5,-11.371,0,0.0374,0.00399,0.86,0.091,0.0509,124.001,4,minimal-techno


### Canciones tipo *mix*

La pista anterior corresponde a un *mix*, lo cual no encaja con el enfoque del proyecto (canciones individuales).

Es probable que existan más casos similares, así que exploraremos con más detalle la distribución de `duration_ms`.


In [13]:
(df_clean['duration_ms']/60000).describe()

count    89583.000000
mean         3.820747
std          1.862646
min          0.263333
25%          2.886000
50%          3.556433
75%          4.406000
max         87.288250
Name: duration_ms, dtype: float64

### Distribución de la duración

En general, los datos están bien distribuidos y la canción de 87 minutos parece un **dato atípico**.

Para confirmarlo, contaremos cuántas canciones superan los **12 minutos** de duración.


In [14]:
print((df_clean["duration_ms"]>720000).sum())

274


### Eliminación de canciones largas

**274** canciones superan los 12 minutos — una cantidad insignificante frente a las ~89.000 del dataset.

**Decisión:** eliminar las canciones con duración mayor a 12 minutos.


In [15]:
df_clean = df_clean[df_clean['duration_ms'] < 720000]
print(f"Filas después de eliminar duraciones > 12 min: {len(df_clean)}")

Filas después de eliminar duraciones > 12 min: 89309


### Revisión de `time_signature`

Según la documentación del dataset, los valores válidos de `time_signature` suelen estar entre **3** y **7**.

Revisaremos cuántos registros tienen valor **0**.


In [16]:
print(f"Canciones con time_signature = 0: {(df_clean['time_signature'] == 0).sum()}")
df_clean['time_signature'].value_counts().sort_index()

Canciones con time_signature = 0: 5


time_signature
0        5
1      841
3     7568
4    79314
5     1581
Name: count, dtype: int64

### Eliminación de `time_signature = 0`

Solo hay **5** registros con `time_signature = 0`, por lo que la mejor decisión es eliminarlos.


In [17]:
df_clean = df_clean[df_clean['time_signature'] != 0]
print(f"Filas después de eliminar time_signature=0: {len(df_clean)}")

Filas después de eliminar time_signature=0: 89304


Finalmente vamos a guardarla el dataset limpio en el csv.

In [18]:
df_clean.to_csv('../data/spotify_tracks_clean.csv', index=False)